# 06 — Pre-POSet EDA Checks

This notebook comes after:

`05_Master_Dataset_Build.ipynb`

Purpose:

Prepare the master country-shock structural snapshot for POSet analysis.

This notebook does:

- read the master country-shock structural snapshot;
- create direction-aligned variables where **higher = better**;
- keep recovery outcomes separate as validation-only variables;
- check missingness by shock and candidate variable;
- check correlation and redundancy among candidate ordering variables;
- create candidate variable scorecards;
- create country coverage scorecards using the explicit baseline variable set;
- recommend defensible baseline and sensitivity variable sets;
- export analysis-ready candidate files.

This notebook does **not**:

- run POSet analysis;
- create final Hasse diagrams;
- create a final scalar Economic Resilience Index;
- use recovery as an ordering variable;
- impute missing values;
- make irreversible sample restrictions.

Main guardrail:

`Recovery`, `Recovery_2007`, and `Recovery_2019` are validation outcomes only.  
They must not be included in POSet ordering variables.

## v2 corrections

This version fixes two issues found after the first diagnostic run:

1. Shock-specific volatility variables are evaluated only for their matching shock window.
2. Country coverage for the baseline set uses the explicit 6-variable baseline set, not every variable whose metadata priority says `baseline_candidate`.


In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 350)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

MASTER_SNAPSHOT_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "Master_Dataset"
    / "master_structural_snapshot_by_shock.csv"
)

MASTER_PANEL_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "Master_Dataset"
    / "master_country_year_panel.csv"
)

MASTER_DICTIONARY_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "Master_Dataset"
    / "master_country_year_panel_data_dictionary.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "06_Pre_POSet_EDA_Checks"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Master snapshot file:", MASTER_SNAPSHOT_FILE.resolve())
print("Master panel file:", MASTER_PANEL_FILE.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())

Run ID: 20260622_051658
Master snapshot file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Master_Dataset\master_structural_snapshot_by_shock.csv
Master panel file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Master_Dataset\master_country_year_panel.csv
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\06_Pre_POSet_EDA_Checks


In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

SHOCK_IDS = ["2007", "2019"]

VALIDATION_OUTCOME_COLUMNS = [
    "Recovery",
    "Recovery_2007",
    "Recovery_2019",
]

# Direction alignment rules:
# - source_column: variable in master_structural_snapshot_by_shock.csv
# - aligned_column: output variable with higher = better
# - direction: already_positive or lower_is_better
# - transform: identity or negative
#
# Important:
# This notebook creates both raw direction-aligned variables and within-shock 0-100 scaled variables.
# For POSet, the ordinal discretization later can use either raw aligned variables or scaled versions.

BASE_DIRECTION_RULES = [
    {
        "source_column": "rd_gdp",
        "aligned_column": "rd_intensity",
        "concept": "R&D intensity",
        "domain": "innovation_capacity",
        "direction": "already_positive",
        "transform": "identity",
        "priority": "baseline_candidate",
        "reason": "R&D/GDP captures innovation capacity and is already higher = better.",
    },
    {
        "source_column": "tertiary_education_25_64",
        "aligned_column": "human_capital_tertiary",
        "concept": "Adult tertiary educational attainment",
        "domain": "human_capital_capacity",
        "direction": "already_positive",
        "transform": "identity",
        "priority": "baseline_candidate",
        "reason": "Adult tertiary attainment captures accumulated human-capital stock.",
    },
    {
        "source_column": "productivity_gdp_per_hour",
        "aligned_column": "productivity_capacity",
        "concept": "GDP per hour worked",
        "domain": "productive_capacity",
        "direction": "already_positive",
        "transform": "identity",
        "priority": "review_candidate",
        "reason": "Productivity captures economic efficiency, but may overlap with broader development/governance.",
    },
    {
        "source_column": "wgi_mahalanobis_ideal_score_full_wgi",
        "aligned_column": "governance_capacity",
        "concept": "Governance capacity",
        "domain": "institutional_capacity",
        "direction": "already_positive",
        "transform": "identity",
        "priority": "baseline_candidate",
        "reason": "Derived WGI governance composite summarizes institutional capacity. It is project-derived, not official WGI.",
    },
    {
        "source_column": "public_debt_gdp_canonical",
        "aligned_column": "debt_capacity",
        "concept": "Public debt capacity",
        "domain": "fiscal_capacity",
        "direction": "lower_is_better",
        "transform": "negative",
        "priority": "baseline_candidate",
        "reason": "Lower public debt/GDP indicates greater fiscal space; converted so higher = better.",
    },
    {
        "source_column": "unemployment_rate",
        "aligned_column": "employment_strength",
        "concept": "Employment strength",
        "domain": "labour_market_capacity",
        "direction": "lower_is_better",
        "transform": "negative",
        "priority": "baseline_candidate",
        "reason": "Lower unemployment indicates stronger labour-market conditions; converted so higher = better.",
    },
    {
        "source_column": "energy_import_dependency_raw",
        "aligned_column": "energy_security",
        "concept": "Energy security",
        "domain": "external_energy_resilience",
        "direction": "lower_is_better",
        "transform": "negative",
        "priority": "baseline_candidate",
        "reason": "Lower energy import dependency means higher energy security; net exporters can have negative dependency and are naturally strong after inversion.",
    },
    {
        "source_column": "inflation_cpi",
        "aligned_column": "price_stability_level",
        "concept": "Price stability level",
        "domain": "macro_stability",
        "direction": "lower_abs_is_better",
        "transform": "negative_absolute",
        "priority": "sensitivity_candidate",
        "reason": "Inflation level is converted via negative absolute inflation. Use carefully because inflation volatility may be preferable.",
    },
]

# Preferred volatility/stability candidates from notebook 04.
# These are shock-specific columns and exist only in the structural snapshot.
VOLATILITY_DIRECTION_RULE_PATTERNS = [
    {
        "source_regex": r"^gdp_growth_stability_negative_std_pre_(2007|2019)$",
        "aligned_base": "gdp_growth_stability",
        "concept": "GDP growth stability",
        "domain": "macro_stability",
        "priority": "baseline_candidate",
        "reason": "Pre-shock GDP growth volatility converted to stability. Higher = more stable.",
    },
    {
        "source_regex": r"^inflation_stability_negative_std_pre_(2007|2019)$",
        "aligned_base": "inflation_stability",
        "concept": "Inflation stability",
        "domain": "macro_stability",
        "priority": "review_candidate",
        "reason": "Pre-shock inflation volatility converted to stability. Check redundancy with inflation level.",
    },
    {
        "source_regex": r"^unemployment_stability_negative_std_pre_(2007|2019)$",
        "aligned_base": "unemployment_stability",
        "concept": "Unemployment stability",
        "domain": "labour_market_capacity",
        "priority": "review_candidate",
        "reason": "Pre-shock unemployment volatility converted to stability. May overlap with employment strength.",
    },
    {
        "source_regex": r"^productivity_level_stability_negative_std_pre_(2007|2019)$",
        "aligned_base": "productivity_stability",
        "concept": "Productivity stability",
        "domain": "productive_capacity",
        "priority": "sensitivity_candidate",
        "reason": "Productivity level volatility can mix instability and trend growth; use as sensitivity candidate.",
    },
]

# A preliminary analysis sample rule for diagnostics only.
# Explicit baseline set used for country coverage and main POSet preparation.
# Shock-specific GDP growth stability is kept as sensitivity, not part of the generic baseline count.
EXPLICIT_BASELINE_6_VARIABLES = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
    "governance_capacity_score_0_100",
]

# Review sample threshold. This is not the final complete-case POSet sample.
MIN_NON_MISSING_BASELINE_CANDIDATES = 4

print("Base direction rules:")
display(pd.DataFrame(BASE_DIRECTION_RULES))

print("Validation-only outcome columns:")
display(pd.DataFrame({"column": VALIDATION_OUTCOME_COLUMNS}))

Base direction rules:


,source_column,aligned_column,concept,domain,direction,transform,priority,reason
0,rd_gdp,rd_intensity,R&D intensity,innovation_capacity,already_positive,identity,baseline_candidate,R&D/GDP captures innovation capacity and is al...
1,tertiary_education_25_64,human_capital_tertiary,Adult tertiary educational attainment,human_capital_capacity,already_positive,identity,baseline_candidate,Adult tertiary attainment captures accumulated...
2,productivity_gdp_per_hour,productivity_capacity,GDP per hour worked,productive_capacity,already_positive,identity,review_candidate,"Productivity captures economic efficiency, but..."
3,wgi_mahalanobis_ideal_score_full_wgi,governance_capacity,Governance capacity,institutional_capacity,already_positive,identity,baseline_candidate,Derived WGI governance composite summarizes in...
4,public_debt_gdp_canonical,debt_capacity,Public debt capacity,fiscal_capacity,lower_is_better,negative,baseline_candidate,Lower public debt/GDP indicates greater fiscal...
5,unemployment_rate,employment_strength,Employment strength,labour_market_capacity,lower_is_better,negative,baseline_candidate,Lower unemployment indicates stronger labour-m...
6,energy_import_dependency_raw,energy_security,Energy security,external_energy_resilience,lower_is_better,negative,baseline_candidate,Lower energy import dependency means higher en...
7,inflation_cpi,price_stability_level,Price stability level,macro_stability,lower_abs_is_better,negative_absolute,sensitivity_candidate,Inflation level is converted via negative abso...


Validation-only outcome columns:


,column
0,Recovery
1,Recovery_2007
2,Recovery_2019


In [3]:
# ------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------

def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def clean_country_shock(df):
    out = df.copy()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.strip()

    if "analysis_year" in out.columns:
        out["analysis_year"] = pd.to_numeric(out["analysis_year"], errors="coerce")
        out = out.dropna(subset=["analysis_year"]).copy()
        out["analysis_year"] = out["analysis_year"].astype(int)

    return out


def min_max_scale_within_group(series):
    values = pd.to_numeric(series, errors="coerce")
    min_value = values.min(skipna=True)
    max_value = values.max(skipna=True)

    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(np.nan, index=series.index)

    if np.isclose(min_value, max_value):
        return pd.Series(100.0, index=series.index)

    return 100.0 * (values - min_value) / (max_value - min_value)


def apply_direction_transform(series, transform):
    values = pd.to_numeric(series, errors="coerce")

    if transform == "identity":
        return values

    if transform == "negative":
        return -values

    if transform == "negative_absolute":
        return -values.abs()

    raise ValueError(f"Unknown transform: {transform}")


def classify_missingness_rate(rate):
    if rate >= 0.90:
        return "strong_coverage"

    if rate >= 0.75:
        return "usable_review"

    if rate >= 0.50:
        return "weak_review"

    return "poor_coverage"


def classify_abs_correlation(abs_corr):
    if pd.isna(abs_corr):
        return "not_available"

    if abs_corr >= 0.90:
        return "very_high_redundancy_risk"

    if abs_corr >= 0.80:
        return "high_redundancy_risk"

    if abs_corr >= 0.70:
        return "moderate_redundancy_risk"

    return "lower_redundancy_risk"


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))

In [4]:
# ------------------------------------------------------
# LOAD MASTER STRUCTURAL SNAPSHOT
# ------------------------------------------------------

require_file(MASTER_SNAPSHOT_FILE, "Master structural snapshot file")

snapshot = pd.read_csv(MASTER_SNAPSHOT_FILE)
snapshot = clean_country_shock(snapshot)

required_columns = {"Country", "shock_id", "analysis_year", "Recovery"}
missing_required_columns = required_columns - set(snapshot.columns)

if missing_required_columns:
    raise ValueError(f"Master snapshot missing required columns: {missing_required_columns}")

if "country_name" not in snapshot.columns:
    snapshot["country_name"] = snapshot["Country"]

input_snapshot_columns = pd.DataFrame({
    "column": snapshot.columns,
    "dtype": [str(snapshot[col].dtype) for col in snapshot.columns],
    "non_missing": [snapshot[col].notna().sum() for col in snapshot.columns],
    "missing": [snapshot[col].isna().sum() for col in snapshot.columns],
})

input_snapshot_columns.to_csv(
    DIAGNOSTICS_DIR / "input_snapshot_columns.csv",
    index=False,
)

snapshot_basic_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "timestamp": RUN_TIMESTAMP,
    "rows": len(snapshot),
    "countries": snapshot["Country"].nunique(),
    "shock_ids": ",".join(sorted(snapshot["shock_id"].unique())),
    "analysis_years": ",".join(map(str, sorted(snapshot["analysis_year"].unique()))),
    "columns": len(snapshot.columns),
}])

snapshot_basic_summary.to_csv(
    DIAGNOSTICS_DIR / "snapshot_basic_summary.csv",
    index=False,
)

print("Snapshot rows:", len(snapshot))
print("Countries:", snapshot["Country"].nunique())
print("Shock IDs:", sorted(snapshot["shock_id"].unique()))
print("Analysis years:", sorted(snapshot["analysis_year"].unique()))
print("Columns:", len(snapshot.columns))

display(snapshot.head())

Snapshot rows: 300
Countries: 162
Shock IDs: ['2007', '2019']
Analysis years: [2007, 2019]
Columns: 150


,Country,country_name,shock_id,analysis_year,pre_shock_window,Recovery,recovery_outcome_column,Year,rd_gdp,inflation_cpi,unemployment_rate,tertiary_education_25_64,productivity_gdp_per_hour,gdp_growth,energy_import_dependency_raw,public_debt_gdp_canonical,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_va_estimate,wgi_pv_estimate,wgi_ge_estimate,wgi_rq_estimate,wgi_rl_estimate,wgi_cc_estimate,wgi_complete_six_selected_dimensions,wgi_composite_input_type,wgi_euclidean_distance_to_ideal_selected_space,wgi_euclidean_ideal_score,wgi_mahalanobis_distance_to_ideal_full_wgi,wgi_mahalanobis_ideal_score_full_wgi,Recovery_2007,Recovery_2019,Recovery_2007_recovery_status,Recovery_2007_shock_detection_status,Recovery_2007_first_negative_growth_year,Recovery_2007_baseline_year,Recovery_2007_baseline_available_in_growth_file,Recovery_2007_recovery_year,Recovery_2007_years_from_baseline_to_recovery,Recovery_2007_years_after_first_negative_to_recovery,Recovery_2007_last_index_year,Recovery_2007_last_index_value,Recovery_2019_recovery_status,Recovery_2019_shock_detection_status,Recovery_2019_first_negative_growth_year,Recovery_2019_baseline_year,Recovery_2019_baseline_available_in_growth_file,Recovery_2019_recovery_year,Recovery_2019_years_from_baseline_to_recovery,Recovery_2019_years_after_first_negative_to_recovery,Recovery_2019_last_index_year,Recovery_2019_last_index_value,country_name_core,country_name_wgi,country_name_recovery,shock_label,window_start,window_end,gdp_growth_volatility_std_pre_2007,inflation_volatility_std_pre_2007,productivity_level_volatility_std_pre_2007,unemployment_volatility_std_pre_2007,gdp_growth_volatility_mad_pre_2007,inflation_volatility_mad_pre_2007,productivity_level_volatility_mad_pre_2007,unemployment_volatility_mad_pre_2007,gdp_growth_stability_negative_std_pre_2007,inflation_stability_negative_std_pre_2007,productivity_level_stability_negative_std_pre_2007,unemployment_stability_negative_std_pre_2007,gdp_growth_stability_negative_mad_pre_2007,inflation_stability_negative_mad_pre_2007,productivity_level_stability_negative_mad_pre_2007,unemployment_stability_negative_mad_pre_2007,gdp_growth_stability_std_0_100_pre_2007,inflation_stability_std_0_100_pre_2007,productivity_level_stability_std_0_100_pre_2007,unemployment_stability_std_0_100_pre_2007,gdp_growth_stability_mad_0_100_pre_2007,inflation_stability_mad_0_100_pre_2007,productivity_level_stability_mad_0_100_pre_2007,unemployment_stability_mad_0_100_pre_2007,gdp_growth_mean_value_pre_2007,inflation_mean_value_pre_2007,productivity_level_mean_value_pre_2007,unemployment_mean_value_pre_2007,gdp_growth_median_value_pre_2007,inflation_median_value_pre_2007,productivity_level_median_value_pre_2007,unemployment_median_value_pre_2007,gdp_growth_change_first_to_last_pre_2007,inflation_change_first_to_last_pre_2007,productivity_level_change_first_to_last_pre_2007,unemployment_change_first_to_last_pre_2007,gdp_growth_years_available_pre_2007,inflation_years_available_pre_2007,productivity_level_years_available_pre_2007,unemployment_years_available_pre_2007,gdp_growth_coverage_rate_in_window_pre_2007,inflation_coverage_rate_in_window_pre_2007,productivity_level_coverage_rate_in_window_pre_2007,unemployment_coverage_rate_in_window_pre_2007,gdp_growth_volatility_std_pre_2019,inflation_volatility_std_pre_2019,productivity_level_volatility_std_pre_2019,unemployment_volatility_std_pre_2019,gdp_growth_volatility_mad_pre_2019,inflation_volatility_mad_pre_2019,productivity_level_volatility_mad_pre_2019,unemployment_volatility_mad_pre_2019,gdp_growth_stability_negative_std_pre_2019,inflation_stability_negative_std_pre_2019,productivity_level_stability_negative_std_pre_2019,unemployment_stability_negative_std_pre_2019,gdp_growth_stability_negative_mad_pre_2019,inflation_stability_negative_mad_pre_2019,productivity_level_stability_negative_mad_pre_2019,unemployment_stability_negative_mad_pre_2019,gdp_growth_stability_std_0_100_pre_2019,inflati

In [5]:
# ------------------------------------------------------
# VALIDATION LEAKAGE GUARDRAIL
# ------------------------------------------------------

validation_cols_present = [
    col for col in VALIDATION_OUTCOME_COLUMNS
    if col in snapshot.columns
]

validation_guardrail = pd.DataFrame([
    {
        "column": col,
        "role": "validation_outcome_only",
        "allowed_in_poset_ordering_variables": False,
        "note": "Do not use recovery outcomes as ordering variables.",
    }
    for col in validation_cols_present
])

validation_guardrail.to_csv(
    DIAGNOSTICS_DIR / "validation_outcome_guardrail.csv",
    index=False,
)

display(validation_guardrail)

,column,role,allowed_in_poset_ordering_variables,note
0,Recovery,validation_outcome_only,False,Do not use recovery outcomes as ordering varia...
1,Recovery_2007,validation_outcome_only,False,Do not use recovery outcomes as ordering varia...
2,Recovery_2019,validation_outcome_only,False,Do not use recovery outcomes as ordering varia...


In [6]:
# ------------------------------------------------------
# CREATE DIRECTION-ALIGNED VARIABLES
# ------------------------------------------------------

aligned = snapshot.copy()

alignment_metadata_rows = []
sanity_rows = []

# Base structural variables.
for rule in BASE_DIRECTION_RULES:
    source_col = rule["source_column"]
    aligned_col = rule["aligned_column"]

    if source_col not in aligned.columns:
        alignment_metadata_rows.append({
            **rule,
            "source_column_found": False,
            "aligned_raw_column": np.nan,
            "aligned_scaled_column": np.nan,
            "note": "Source column not found in snapshot.",
        })
        continue

    raw_col = f"{aligned_col}_aligned_raw"
    scaled_col = f"{aligned_col}_score_0_100"

    aligned[raw_col] = apply_direction_transform(aligned[source_col], rule["transform"])
    aligned[scaled_col] = np.nan

    for shock_id, idx in aligned.groupby("shock_id").groups.items():
        idx = list(idx)
        aligned.loc[idx, scaled_col] = min_max_scale_within_group(aligned.loc[idx, raw_col])

    alignment_metadata_rows.append({
        **rule,
        "source_column_found": True,
        "aligned_raw_column": raw_col,
        "aligned_scaled_column": scaled_col,
        "note": "Created successfully.",
    })

    sanity_rows.append({
        "source_column": source_col,
        "aligned_column": aligned_col,
        "aligned_raw_column": raw_col,
        "aligned_scaled_column": scaled_col,
        "source_non_missing": int(aligned[source_col].notna().sum()),
        "aligned_non_missing": int(aligned[raw_col].notna().sum()),
        "source_min": pd.to_numeric(aligned[source_col], errors="coerce").min(),
        "source_median": pd.to_numeric(aligned[source_col], errors="coerce").median(),
        "source_max": pd.to_numeric(aligned[source_col], errors="coerce").max(),
        "aligned_min": pd.to_numeric(aligned[raw_col], errors="coerce").min(),
        "aligned_median": pd.to_numeric(aligned[raw_col], errors="coerce").median(),
        "aligned_max": pd.to_numeric(aligned[raw_col], errors="coerce").max(),
        "scaled_min": pd.to_numeric(aligned[scaled_col], errors="coerce").min(),
        "scaled_median": pd.to_numeric(aligned[scaled_col], errors="coerce").median(),
        "scaled_max": pd.to_numeric(aligned[scaled_col], errors="coerce").max(),
    })

# Shock-specific volatility/stability variables.
for pattern_rule in VOLATILITY_DIRECTION_RULE_PATTERNS:
    source_regex = pattern_rule["source_regex"]

    for source_col in snapshot.columns:
        match = re.match(source_regex, source_col)

        if not match:
            continue

        shock_id_from_col = match.group(1)
        aligned_base = pattern_rule["aligned_base"]
        aligned_col = f"{aligned_base}_pre_{shock_id_from_col}"
        raw_col = f"{aligned_col}_aligned_raw"
        scaled_col = f"{aligned_col}_score_0_100"

        aligned[raw_col] = pd.to_numeric(aligned[source_col], errors="coerce")
        aligned[scaled_col] = np.nan

        # Only scale within the matching shock rows.
        matching_idx = aligned.index[aligned["shock_id"].astype(str) == str(shock_id_from_col)].tolist()
        aligned.loc[matching_idx, scaled_col] = min_max_scale_within_group(aligned.loc[matching_idx, raw_col])

        # Force non-matching shock rows to missing for safety.
        non_matching_idx = aligned.index[aligned["shock_id"].astype(str) != str(shock_id_from_col)].tolist()
        aligned.loc[non_matching_idx, [raw_col, scaled_col]] = np.nan

        alignment_metadata_rows.append({
            "source_column": source_col,
            "aligned_column": aligned_col,
            "concept": pattern_rule["concept"],
            "domain": pattern_rule["domain"],
            "direction": "already_positive_after_volatility_notebook",
            "transform": "identity",
            "priority": pattern_rule["priority"],
            "reason": pattern_rule["reason"],
            "source_column_found": True,
            "aligned_raw_column": raw_col,
            "aligned_scaled_column": scaled_col,
            "note": "Shock-specific stability feature created successfully.",
        })

        sanity_rows.append({
            "source_column": source_col,
            "aligned_column": aligned_col,
            "aligned_raw_column": raw_col,
            "aligned_scaled_column": scaled_col,
            "source_non_missing": int(aligned[source_col].notna().sum()),
            "aligned_non_missing": int(aligned[raw_col].notna().sum()),
            "source_min": pd.to_numeric(aligned[source_col], errors="coerce").min(),
            "source_median": pd.to_numeric(aligned[source_col], errors="coerce").median(),
            "source_max": pd.to_numeric(aligned[source_col], errors="coerce").max(),
            "aligned_min": pd.to_numeric(aligned[raw_col], errors="coerce").min(),
            "aligned_median": pd.to_numeric(aligned[raw_col], errors="coerce").median(),
            "aligned_max": pd.to_numeric(aligned[raw_col], errors="coerce").max(),
            "scaled_min": pd.to_numeric(aligned[scaled_col], errors="coerce").min(),
            "scaled_median": pd.to_numeric(aligned[scaled_col], errors="coerce").median(),
            "scaled_max": pd.to_numeric(aligned[scaled_col], errors="coerce").max(),
        })

variable_direction_alignment_metadata = pd.DataFrame(alignment_metadata_rows)
direction_alignment_sanity_checks = pd.DataFrame(sanity_rows)

variable_direction_alignment_metadata.to_csv(
    PROCESSED_DIR / "variable_direction_alignment_metadata.csv",
    index=False,
)

variable_direction_alignment_metadata.to_csv(
    DIAGNOSTICS_DIR / "variable_direction_alignment_metadata.csv",
    index=False,
)

direction_alignment_sanity_checks.to_csv(
    DIAGNOSTICS_DIR / "direction_alignment_sanity_checks.csv",
    index=False,
)

aligned.to_csv(
    PROCESSED_DIR / "pre_poset_structural_snapshot_direction_aligned.csv",
    index=False,
)

print("Aligned snapshot rows:", len(aligned))
print("Aligned columns:", len(aligned.columns))

display(variable_direction_alignment_metadata)
display(direction_alignment_sanity_checks)

Aligned snapshot rows: 300
Aligned columns: 182


,source_column,aligned_column,concept,domain,direction,transform,priority,reason,source_column_found,aligned_raw_column,aligned_scaled_column,note
0,rd_gdp,rd_intensity,R&D intensity,innovation_capacity,already_positive,identity,baseline_candidate,R&D/GDP captures innovation capacity and is al...,True,rd_intensity_aligned_raw,rd_intensity_score_0_100,Created successfully.
1,tertiary_education_25_64,human_capital_tertiary,Adult tertiary educational attainment,human_capital_capacity,already_positive,identity,baseline_candidate,Adult tertiary attainment captures accumulated...,True,human_capital_tertiary_aligned_raw,human_capital_tertiary_score_0_100,Created successfully.
2,productivity_gdp_per_hour,productivity_capacity,GDP per hour worked,productive_capacity,already_positive,identity,review_candidate,"Productivity captures economic efficiency, but...",True,productivity_capacity_aligned_raw,productivity_capacity_score_0_100,Created successfully.
3,wgi_mahalanobis_ideal_score_full_wgi,governance_capacity,Governance capacity,institutional_capacity,already_positive,identity,baseline_candidate,Derived WGI governance composite summarizes in...,True,governance_capacity_aligned_raw,governance_capacity_score_0_100,Created successfully.
4,public_debt_gdp_canonical,debt_capacity,Public debt capacity,fiscal_capacity,lower_is_better,negative,baseline_candidate,Lower public debt/GDP indicates greater fiscal...,True,debt_capacity_aligned_raw,debt_capacity_score_0_100,Created successfully.
5,unemployment_rate,employment_strength,Employment strength,labour_market_capacity,lower_is_better,negative,baseline_candidate,Lower unemployment indicates stronger labour-m...,True,employment_strength_aligned_raw,employment_strength_score_0_100,Created successfully.
6,energy_import_dependency_raw,energy_security,Energy security,external_energy_resilience,lower_is_better,negative,baseline_candidate,Lower energy import dependency means higher en...,True,energy_security_aligned_raw,energy_security_score_0_100,Created successfully.
7,inflation_cpi,price_stability_level,Price stability level,macro_stability,lower_abs_is_better,negative_absolute,sensitivity_candidate,Inflation level is converted via negative abso...,True,price_stability_level_aligned_raw,price_stability_level_score_0_100,Created successfully.
8,gdp_growth_stability_negative_std_pre_2007,gdp_growth_stability_pre_2007,GDP growth stability,macro_stability,already_positive_after_volatility_notebook,identity,baseline_candidate,Pre-shock GDP growth volatility converted to s...,True,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_pre_2007_score_0_100,Shock-specific stability feature created succe...
9,gdp_growth_stability_negative_std_pre_2019,gdp_growth_stability_pre_2019,GDP growth stability,macro_stability,already_positive_after_volatility_notebook,identity,baseline_candidate,Pre-shock GDP growth volatility converted to s...,True,gdp_growth_stability_pre_2019_aligned_raw,gdp_growth_stability_pre_2019_score_0_100,Shock-specific stability feature created succe...


,source_column,aligned_column,aligned_raw_column,aligned_scaled_column,source_non_missing,aligned_non_missing,source_min,source_median,source_max,aligned_min,aligned_median,aligned_max,scaled_min,scaled_median,scaled_max
0,rd_gdp,rd_intensity,rd_intensity_aligned_raw,rd_intensity_score_0_100,89,89,0.1831,1.4642,5.7912,0.1831,1.4642,5.7912,0.0000,26.2152,100.0000
1,tertiary_education_25_64,human_capital_tertiary,human_capital_tertiary_aligned_raw,human_capital_tertiary_score_0_100,82,82,6.8958,31.1157,59.0203,6.8958,31.1157,59.0203,0.0000,51.6764,100.0000
2,productivity_gdp_per_hour,productivity_capacity,productivity_capacity_aligned_raw,productivity_capacity_score_0_100,81,81,14.0548,50.1238,118.7455,14.0548,50.1238,118.7455,0.0000,35.1675,100.0000
3,wgi_mahalanobis_ideal_score_full_wgi,governance_capacity,governance_capacity_aligned_raw,governance_capacity_score_0_100,298,298,15.5352,51.9064,81.5177,15.5352,51.9064,81.5177,0.0000,52.4061,100.0000
4,public_debt_gdp_canonical,debt_capacity,debt_capacity_aligned_raw,debt_capacity_score_0_100,149,149,3.9000,45.2981,183.2000,-183.2000,-45.2981,-3.9000,0.0000,72.6109,100.0000
5,unemployment_rate,employment_strength,employment_strength_aligned_raw,employment_strength_score_0_100,101,101,0.7210,5.3580,34.9350,-34.9350,-5.3580,-0.7210,0.0000,86.6510,100.0000
6,energy_import_dependency_raw,energy_security,energy_security_aligned_raw,energy_security_score_0_100,267,267,-1282.7102,27.0601,424.2078,-424.2078,-27.0601,1282.7102,0.0000,24.4474,100.0000
7,inflation_cpi,price_stability_level,price_stability_level_aligned_raw,price_stability_level_score_0_100,95,95,-2.0933,2.4540,53.5483,-53.5483,-2.4540,-0.0000,0.0000,92.7440,100.0000
8,gdp_growth_stability_negative_std_pre_2007,gdp_growth_stability_pre_2007,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_pre_2007_score_0_100,44,44,-4.7569,-1.5573,-0.7130,-4.7569,-1.5573,-0.7130,0.0000,79.1199,100.0000
9,gdp_growth_stability_negative_std_pre_2019,gdp_growth_stability_pre_2019,gdp_growth_stability_pre_2019_aligned_raw,gdp_growth_stability_pre_2019_score_0_100,44,44,-7.9289,-1.2124,-0.3842,-7.9289,-1.2124,-0.3842,0.0000,89.0231,100.0000


In [7]:
# ------------------------------------------------------
# CANDIDATE ORDERING VARIABLE INVENTORY
# ------------------------------------------------------

candidate_rows = []

for _, row in variable_direction_alignment_metadata.iterrows():
    if not bool(row.get("source_column_found", False)):
        continue

    candidate_rows.append({
        "candidate_variable": row["aligned_scaled_column"],
        "candidate_variable_raw": row["aligned_raw_column"],
        "source_column": row["source_column"],
        "concept": row["concept"],
        "domain": row["domain"],
        "priority": row["priority"],
        "direction": "higher_is_better",
        "reason": row["reason"],
        "recommended_use": (
            "main_or_baseline"
            if row["priority"] == "baseline_candidate"
            else "review_or_sensitivity"
        ),
    })

candidate_variable_inventory = pd.DataFrame(candidate_rows)

# Explicitly exclude recovery variables if they were accidentally matched.
candidate_variable_inventory = candidate_variable_inventory[
    ~candidate_variable_inventory["candidate_variable"].isin(VALIDATION_OUTCOME_COLUMNS)
].copy()

candidate_variable_inventory.to_csv(
    PROCESSED_DIR / "candidate_ordering_variable_inventory.csv",
    index=False,
)

candidate_variable_inventory.to_csv(
    DIAGNOSTICS_DIR / "candidate_ordering_variable_inventory.csv",
    index=False,
)

display(candidate_variable_inventory)

,candidate_variable,candidate_variable_raw,source_column,concept,domain,priority,direction,reason,recommended_use
0,rd_intensity_score_0_100,rd_intensity_aligned_raw,rd_gdp,R&D intensity,innovation_capacity,baseline_candidate,higher_is_better,R&D/GDP captures innovation capacity and is al...,main_or_baseline
1,human_capital_tertiary_score_0_100,human_capital_tertiary_aligned_raw,tertiary_education_25_64,Adult tertiary educational attainment,human_capital_capacity,baseline_candidate,higher_is_better,Adult tertiary attainment captures accumulated...,main_or_baseline
2,productivity_capacity_score_0_100,productivity_capacity_aligned_raw,productivity_gdp_per_hour,GDP per hour worked,productive_capacity,review_candidate,higher_is_better,"Productivity captures economic efficiency, but...",review_or_sensitivity
3,governance_capacity_score_0_100,governance_capacity_aligned_raw,wgi_mahalanobis_ideal_score_full_wgi,Governance capacity,institutional_capacity,baseline_candidate,higher_is_better,Derived WGI governance composite summarizes in...,main_or_baseline
4,debt_capacity_score_0_100,debt_capacity_aligned_raw,public_debt_gdp_canonical,Public debt capacity,fiscal_capacity,baseline_candidate,higher_is_better,Lower public debt/GDP indicates greater fiscal...,main_or_baseline
5,employment_strength_score_0_100,employment_strength_aligned_raw,unemployment_rate,Employment strength,labour_market_capacity,baseline_candidate,higher_is_better,Lower unemployment indicates stronger labour-m...,main_or_baseline
6,energy_security_score_0_100,energy_security_aligned_raw,energy_import_dependency_raw,Energy security,external_energy_resilience,baseline_candidate,higher_is_better,Lower energy import dependency means higher en...,main_or_baseline
7,price_stability_level_score_0_100,price_stability_level_aligned_raw,inflation_cpi,Price stability level,macro_stability,sensitivity_candidate,higher_is_better,Inflation level is converted via negative abso...,review_or_sensitivity
8,gdp_growth_stability_pre_2007_score_0_100,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_negative_std_pre_2007,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline
9,gdp_growth_stability_pre_2019_score_0_100,gdp_growth_stability_pre_2019_aligned_raw,gdp_growth_stability_negative_std_pre_2019,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline


In [8]:
# ------------------------------------------------------
# MISSINGNESS BY SHOCK AND CANDIDATE VARIABLE
# ------------------------------------------------------
# v2 correction:
# Shock-specific variables like *_pre_2007 are evaluated only for shock_id = 2007.
# This avoids artificial 0% coverage when the variable is correctly missing for the other shock.

candidate_scaled_cols = candidate_variable_inventory["candidate_variable"].dropna().tolist()
candidate_raw_cols = candidate_variable_inventory["candidate_variable_raw"].dropna().tolist()

def candidate_applicable_to_shock(candidate_variable, shock_id):
    candidate_variable = str(candidate_variable)
    shock_id = str(shock_id)

    pre_match = re.search(r"_pre_(\d{4})", candidate_variable)

    if pre_match:
        return pre_match.group(1) == shock_id

    return True

missingness_rows = []

for shock_id, group in aligned.groupby("shock_id"):
    for _, meta in candidate_variable_inventory.iterrows():
        col = meta["candidate_variable"]

        if col not in group.columns:
            continue

        if not candidate_applicable_to_shock(col, shock_id):
            continue

        values = group[col]
        coverage_rate = values.notna().mean()

        analysis_group = group[group["Recovery"].notna()].copy()
        analysis_values = analysis_group[col] if col in analysis_group.columns else pd.Series(dtype=float)
        analysis_coverage_rate = analysis_values.notna().mean() if len(analysis_group) > 0 else np.nan

        missingness_rows.append({
            "shock_id": shock_id,
            "candidate_variable": col,
            "candidate_variable_raw": meta["candidate_variable_raw"],
            "source_column": meta["source_column"],
            "concept": meta["concept"],
            "domain": meta["domain"],
            "priority": meta["priority"],
            "rows_total": len(group),
            "rows_non_missing": int(values.notna().sum()),
            "countries_non_missing": int(group.loc[values.notna(), "Country"].nunique()),
            "coverage_rate": coverage_rate,
            "coverage_class": classify_missingness_rate(coverage_rate),
            "analysis_rows_total_recovery_available": len(analysis_group),
            "analysis_rows_non_missing": int(analysis_values.notna().sum()) if len(analysis_group) > 0 else 0,
            "analysis_countries_non_missing": int(analysis_group.loc[analysis_values.notna(), "Country"].nunique()) if len(analysis_group) > 0 else 0,
            "analysis_coverage_rate_recovery_sample": analysis_coverage_rate,
            "analysis_coverage_class": classify_missingness_rate(analysis_coverage_rate) if pd.notna(analysis_coverage_rate) else "not_available",
            "min_value": pd.to_numeric(values, errors="coerce").min(),
            "median_value": pd.to_numeric(values, errors="coerce").median(),
            "max_value": pd.to_numeric(values, errors="coerce").max(),
        })

candidate_variable_missingness_by_shock = pd.DataFrame(missingness_rows)

candidate_variable_missingness_by_shock.to_csv(
    DIAGNOSTICS_DIR / "candidate_variable_missingness_by_shock.csv",
    index=False,
)

display(candidate_variable_missingness_by_shock)

,shock_id,candidate_variable,candidate_variable_raw,source_column,concept,domain,priority,rows_total,rows_non_missing,countries_non_missing,coverage_rate,coverage_class,analysis_rows_total_recovery_available,analysis_rows_non_missing,analysis_countries_non_missing,analysis_coverage_rate_recovery_sample,analysis_coverage_class,min_value,median_value,max_value
0,2007,rd_intensity_score_0_100,rd_intensity_aligned_raw,rd_gdp,R&D intensity,innovation_capacity,baseline_candidate,146,44,44,0.3014,poor_coverage,44,38,38,0.8636,usable_review,0.0000,26.3994,100.0000
1,2007,human_capital_tertiary_score_0_100,human_capital_tertiary_aligned_raw,tertiary_education_25_64,Adult tertiary educational attainment,human_capital_capacity,baseline_candidate,146,37,37,0.2534,poor_coverage,44,37,37,0.8409,usable_review,0.0000,53.3734,100.0000
2,2007,productivity_capacity_score_0_100,productivity_capacity_aligned_raw,productivity_gdp_per_hour,GDP per hour worked,productive_capacity,review_candidate,146,40,40,0.2740,poor_coverage,44,37,37,0.8409,usable_review,0.0000,32.2588,100.0000
3,2007,governance_capacity_score_0_100,governance_capacity_aligned_raw,wgi_mahalanobis_ideal_score_full_wgi,Governance capacity,institutional_capacity,baseline_candidate,146,144,144,0.9863,strong_coverage,44,44,44,1.0000,strong_coverage,0.0000,56.2890,100.0000
4,2007,debt_capacity_score_0_100,debt_capacity_aligned_raw,public_debt_gdp_canonical,Public debt capacity,fiscal_capacity,baseline_candidate,146,68,68,0.4658,poor_coverage,44,31,31,0.7045,weak_review,0.0000,65.6830,100.0000
5,2007,employment_strength_score_0_100,employment_strength_aligned_raw,unemployment_rate,Employment strength,labour_market_capacity,baseline_candidate,146,49,49,0.3356,poor_coverage,44,42,42,0.9545,strong_coverage,0.0000,88.9286,100.0000
6,2007,energy_security_score_0_100,energy_security_aligned_raw,energy_import_dependency_raw,Energy security,external_energy_resilience,baseline_candidate,146,130,130,0.8904,usable_review,44,43,43,0.9773,strong_coverage,0.0000,15.8649,100.0000
7,2007,price_stability_level_score_0_100,price_stability_level_aligned_raw,inflation_cpi,Price stability level,macro_stability,sensitivity_candidate,146,47,47,0.3219,poor_coverage,44,44,44,1.0000,strong_coverage,0.0000,71.7361,100.0000
8,2007,gdp_growth_stability_pre_2007_score_0_100,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_negative_std_pre_2007,GDP growth stability,macro_stability,baseline_candidate,146,44,44,0.3014,poor_coverage,44,44,44,1.0000,strong_coverage,0.0000,79.1199,100.0000
9,2007,inflation_stability_pre_2007_score_0_100,inflation_stability_pre_2007_aligned_raw,inflation_stability_negative_std_pre_2007,Inflation stability,macro_stability,review_candidate,146,47,47,0.3219,poor_coverage,44,44,44,1.0000,strong_coverage,0.0000,96.2356,100.0000


In [9]:
# ------------------------------------------------------
# CORRELATION AND REDUNDANCY CHECKS
# ------------------------------------------------------
# v2 correction:
# For each shock, use only variables applicable to that shock.
# Also compute correlations on the recovery-available analysis sample where possible.

correlation_long_rows = []
redundancy_pair_rows = []

for shock_id, group in aligned.groupby("shock_id"):
    active_cols = [
        col for col in candidate_scaled_cols
        if (
            col in group.columns
            and candidate_applicable_to_shock(col, shock_id)
            and group[col].notna().sum() >= 5
        )
    ]

    if len(active_cols) < 2:
        continue

    # Prefer recovery-available countries for redundancy checks because this is the sample
    # that will likely feed validation and most POSet reporting.
    analysis_group = group[group["Recovery"].notna()].copy()

    corr_input = analysis_group[active_cols].copy()

    if len(corr_input.dropna(how="all")) < 5:
        corr_input = group[active_cols].copy()

    corr_matrix = corr_input.corr()

    corr_matrix.to_csv(
        DIAGNOSTICS_DIR / f"candidate_variable_correlation_matrix_shock_{shock_id}.csv"
    )

    for i, col_a in enumerate(active_cols):
        for j, col_b in enumerate(active_cols):
            if j <= i:
                continue

            pair_data = corr_input[[col_a, col_b]].dropna()
            corr = corr_matrix.loc[col_a, col_b]

            meta_a = candidate_variable_inventory[
                candidate_variable_inventory["candidate_variable"] == col_a
            ].iloc[0]

            meta_b = candidate_variable_inventory[
                candidate_variable_inventory["candidate_variable"] == col_b
            ].iloc[0]

            row = {
                "shock_id": shock_id,
                "variable_a": col_a,
                "variable_b": col_b,
                "concept_a": meta_a["concept"],
                "concept_b": meta_b["concept"],
                "domain_a": meta_a["domain"],
                "domain_b": meta_b["domain"],
                "priority_a": meta_a["priority"],
                "priority_b": meta_b["priority"],
                "countries_used_pairwise": len(pair_data),
                "correlation": corr,
                "abs_correlation": abs(corr) if pd.notna(corr) else np.nan,
                "redundancy_class": classify_abs_correlation(abs(corr) if pd.notna(corr) else np.nan),
                "same_domain": meta_a["domain"] == meta_b["domain"],
            }

            correlation_long_rows.append(row)

            if pd.notna(corr) and abs(corr) >= 0.70:
                redundancy_pair_rows.append(row)

candidate_variable_correlation_pairs = pd.DataFrame(correlation_long_rows)
candidate_variable_redundancy_pairs = pd.DataFrame(redundancy_pair_rows)

candidate_variable_correlation_pairs.to_csv(
    DIAGNOSTICS_DIR / "candidate_variable_correlation_pairs.csv",
    index=False,
)

candidate_variable_redundancy_pairs.to_csv(
    DIAGNOSTICS_DIR / "candidate_variable_redundancy_pairs.csv",
    index=False,
)

if candidate_variable_redundancy_pairs.empty:
    display(pd.DataFrame({"message": ["No redundancy pairs with abs(correlation) >= 0.70."]}))
else:
    display(candidate_variable_redundancy_pairs.sort_values(["shock_id", "abs_correlation"], ascending=[True, False]).head(100))

,shock_id,variable_a,variable_b,concept_a,concept_b,domain_a,domain_b,priority_a,priority_b,countries_used_pairwise,correlation,abs_correlation,redundancy_class,same_domain
0,2019,price_stability_level_score_0_100,inflation_stability_pre_2019_score_0_100,Price stability level,Inflation stability,macro_stability,macro_stability,sensitivity_candidate,review_candidate,43,0.7857,0.7857,moderate_redundancy_risk,True
1,2019,gdp_growth_stability_pre_2019_score_0_100,productivity_stability_pre_2019_score_0_100,GDP growth stability,Productivity stability,macro_stability,productive_capacity,baseline_candidate,sensitivity_candidate,38,0.7370,0.7370,moderate_redundancy_risk,False


In [10]:
# ------------------------------------------------------
# CANDIDATE VARIABLE SCORECARD
# ------------------------------------------------------
# v2 correction:
# Recommendation uses coverage in the recovery-available analysis sample,
# while still preserving broad-master coverage for transparency.

scorecard = candidate_variable_inventory.copy()

coverage_agg = (
    candidate_variable_missingness_by_shock
    .groupby("candidate_variable")
    .agg(
        min_coverage_rate_broad_master=("coverage_rate", "min"),
        mean_coverage_rate_broad_master=("coverage_rate", "mean"),
        min_countries_non_missing_broad_master=("countries_non_missing", "min"),
        min_analysis_coverage_rate=("analysis_coverage_rate_recovery_sample", "min"),
        mean_analysis_coverage_rate=("analysis_coverage_rate_recovery_sample", "mean"),
        min_analysis_countries_non_missing=("analysis_countries_non_missing", "min"),
        shocks_available=("shock_id", "nunique"),
    )
    .reset_index()
)

scorecard = scorecard.merge(
    coverage_agg,
    on="candidate_variable",
    how="left",
)

# Redundancy risk: maximum absolute correlation with another variable.
if not candidate_variable_correlation_pairs.empty:
    corr_for_a = candidate_variable_correlation_pairs[
        ["variable_a", "abs_correlation", "redundancy_class"]
    ].rename(columns={"variable_a": "candidate_variable"})

    corr_for_b = candidate_variable_correlation_pairs[
        ["variable_b", "abs_correlation", "redundancy_class"]
    ].rename(columns={"variable_b": "candidate_variable"})

    corr_combined = pd.concat([corr_for_a, corr_for_b], ignore_index=True)

    redundancy_agg = (
        corr_combined
        .groupby("candidate_variable")
        .agg(max_abs_correlation=("abs_correlation", "max"))
        .reset_index()
    )

    redundancy_agg["max_redundancy_class"] = redundancy_agg["max_abs_correlation"].apply(classify_abs_correlation)
else:
    redundancy_agg = pd.DataFrame(columns=["candidate_variable", "max_abs_correlation", "max_redundancy_class"])

scorecard = scorecard.merge(
    redundancy_agg,
    on="candidate_variable",
    how="left",
)

def recommend_variable(row):
    sample_cov = row.get("min_analysis_coverage_rate", np.nan)

    if pd.isna(sample_cov):
        return "weak_or_incomplete_review"

    if row["priority"] == "baseline_candidate" and sample_cov >= 0.75:
        if row.get("max_abs_correlation", 0) >= 0.90:
            return "baseline_candidate_but_high_redundancy_review"
        return "recommended_baseline_pool"

    if row["priority"] == "review_candidate" and sample_cov >= 0.75:
        return "review_candidate"

    if row["priority"] == "sensitivity_candidate" and sample_cov >= 0.75:
        return "sensitivity_candidate"

    return "weak_or_incomplete_review"

scorecard["pre_poset_recommendation"] = scorecard.apply(recommend_variable, axis=1)

scorecard["warning"] = np.where(
    scorecard["candidate_variable"].str.contains("Recovery", case=False, na=False),
    "ERROR: recovery variable should not be candidate",
    ""
)

candidate_variable_scorecard = scorecard.sort_values(
    ["pre_poset_recommendation", "domain", "candidate_variable"]
).reset_index(drop=True)

candidate_variable_scorecard.to_csv(
    PROCESSED_DIR / "candidate_variable_scorecard.csv",
    index=False,
)

candidate_variable_scorecard.to_csv(
    DIAGNOSTICS_DIR / "candidate_variable_scorecard.csv",
    index=False,
)

display(candidate_variable_scorecard)

,candidate_variable,candidate_variable_raw,source_column,concept,domain,priority,direction,reason,recommended_use,min_coverage_rate_broad_master,mean_coverage_rate_broad_master,min_countries_non_missing_broad_master,min_analysis_coverage_rate,mean_analysis_coverage_rate,min_analysis_countries_non_missing,shocks_available,max_abs_correlation,max_redundancy_class,pre_poset_recommendation,warning
0,energy_security_score_0_100,energy_security_aligned_raw,energy_import_dependency_raw,Energy security,external_energy_resilience,baseline_candidate,higher_is_better,Lower energy import dependency means higher en...,main_or_baseline,0.8896,0.8900,130,0.9767,0.9770,42,2,0.3162,lower_redundancy_risk,recommended_baseline_pool,
1,human_capital_tertiary_score_0_100,human_capital_tertiary_aligned_raw,tertiary_education_25_64,Adult tertiary educational attainment,human_capital_capacity,baseline_candidate,higher_is_better,Adult tertiary attainment captures accumulated...,main_or_baseline,0.2534,0.2728,37,0.8409,0.8972,37,2,0.6869,lower_redundancy_risk,recommended_baseline_pool,
2,rd_intensity_score_0_100,rd_intensity_aligned_raw,rd_gdp,R&D intensity,innovation_capacity,baseline_candidate,higher_is_better,R&D/GDP captures innovation capacity and is al...,main_or_baseline,0.2922,0.2968,44,0.8636,0.8737,38,2,0.6869,lower_redundancy_risk,recommended_baseline_pool,
3,governance_capacity_score_0_100,governance_capacity_aligned_raw,wgi_mahalanobis_ideal_score_full_wgi,Governance capacity,institutional_capacity,baseline_candidate,higher_is_better,Derived WGI governance composite summarizes in...,main_or_baseline,0.9863,0.9932,144,1.0000,1.0000,43,2,0.5497,lower_redundancy_risk,recommended_baseline_pool,
4,employment_strength_score_0_100,employment_strength_aligned_raw,unemployment_rate,Employment strength,labour_market_capacity,baseline_candidate,higher_is_better,Lower unemployment indicates stronger labour-m...,main_or_baseline,0.3356,0.3366,49,0.9545,0.9656,42,2,0.4342,lower_redundancy_risk,recommended_baseline_pool,
5,gdp_growth_stability_pre_2007_score_0_100,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_negative_std_pre_2007,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline,0.3014,0.3014,44,1.0000,1.0000,44,1,0.6874,lower_redundancy_risk,recommended_baseline_pool,
6,gdp_growth_stability_pre_2019_score_0_100,gdp_growth_stability_pre_2019_aligned_raw,gdp_growth_stability_negative_std_pre_2019,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline,0.2857,0.2857,44,1.0000,1.0000,43,1,0.7370,moderate_redundancy_risk,recommended_baseline_pool,
7,unemployment_stability_pre_2007_score_0_100,unemployment_stability_pre_2007_aligned_raw,unemployment_stability_negative_std_pre_2007,Unemployment stability,labour_market_capacity,review_candidate,higher_is_better,Pre-shock unemployment volatility converted to...,review_or_sensitivity,0.3288,0.3288,48,0.9545,0.9545,42,1,0.4128,lower_redundancy_risk,review_candidate,
8,unemployment_stability_pre_2019_score_0_100,unemployment_stability_pre_2019_aligned_raw,unemployment_stability_negative_std_pre_2019,Unemployment stability,labour_market_capacity,review_candidate,higher_is_better,Pre-shock unemployment volatility converted to...,review_or_sensitivity,0.3312,0.3312,51,0.9535,0.9535,41,1,0.6411,lower_redundancy_risk,review_candidate,
9,inflation_stability_pre_2007_score_0_100,inflation_stability_pre_2007_aligned_raw,inflation_stability_negative_std_pre_2007,Inflation stability,macro_stability,review_candidate,higher_is_better,Pre-shock inflation volatility converted to st...,review_or_sensitivity,0.3219,0.3219,47,1.0000,1.0000,44,1,0.6874,lower_redundancy_risk,review_candidate,


In [11]:
# ------------------------------------------------------
# COUNTRY COVERAGE SCORECARD
# ------------------------------------------------------
# v2 correction:
# Use explicit baseline 6-variable set only.
# Do not include shock-specific GDP stability in the generic baseline count.

baseline_candidate_cols = [
    col for col in EXPLICIT_BASELINE_6_VARIABLES
    if col in aligned.columns
]

country_coverage_rows = []

for shock_id, group in aligned.groupby("shock_id"):
    for _, row in group.iterrows():
        values = row[baseline_candidate_cols]

        non_missing_count = int(values.notna().sum())
        coverage_rate = non_missing_count / len(baseline_candidate_cols) if baseline_candidate_cols else np.nan

        country_coverage_rows.append({
            "Country": row["Country"],
            "country_name": row.get("country_name", row["Country"]),
            "shock_id": shock_id,
            "analysis_year": row.get("analysis_year", np.nan),
            "Recovery": row.get("Recovery", np.nan),
            "baseline_candidate_variables_total": len(baseline_candidate_cols),
            "baseline_candidate_variables_available": non_missing_count,
            "baseline_candidate_coverage_rate": coverage_rate,
            "baseline_complete_case": non_missing_count == len(baseline_candidate_cols),
            "has_recovery": pd.notna(row.get("Recovery", np.nan)),
            "recommended_for_pre_poset_review": (
                non_missing_count >= MIN_NON_MISSING_BASELINE_CANDIDATES
                and pd.notna(row.get("Recovery", np.nan))
            ),
            "recommended_for_baseline_complete_case_poset": (
                non_missing_count == len(baseline_candidate_cols)
                and pd.notna(row.get("Recovery", np.nan))
            ),
        })

country_coverage_scorecard = pd.DataFrame(country_coverage_rows)

country_coverage_scorecard.to_csv(
    PROCESSED_DIR / "country_coverage_scorecard.csv",
    index=False,
)

country_coverage_scorecard.to_csv(
    DIAGNOSTICS_DIR / "country_coverage_scorecard.csv",
    index=False,
)

recommended_analysis_sample_by_shock = country_coverage_scorecard[
    country_coverage_scorecard["recommended_for_pre_poset_review"]
].copy()

baseline_complete_case_sample_by_shock = country_coverage_scorecard[
    country_coverage_scorecard["recommended_for_baseline_complete_case_poset"]
].copy()

recommended_analysis_sample_by_shock.to_csv(
    PROCESSED_DIR / "recommended_analysis_sample_by_shock.csv",
    index=False,
)

recommended_analysis_sample_by_shock.to_csv(
    DIAGNOSTICS_DIR / "recommended_analysis_sample_by_shock.csv",
    index=False,
)

baseline_complete_case_sample_by_shock.to_csv(
    PROCESSED_DIR / "baseline_complete_case_sample_by_shock.csv",
    index=False,
)

baseline_complete_case_sample_by_shock.to_csv(
    DIAGNOSTICS_DIR / "baseline_complete_case_sample_by_shock.csv",
    index=False,
)

print("Explicit baseline variables:", baseline_candidate_cols)
print("Review sample by shock:")
display(recommended_analysis_sample_by_shock.groupby("shock_id").size().reset_index(name="countries"))
print("Baseline complete-case sample by shock:")
display(baseline_complete_case_sample_by_shock.groupby("shock_id").size().reset_index(name="countries"))

display(country_coverage_scorecard.sort_values(["shock_id", "baseline_candidate_variables_available", "Country"], ascending=[True, False, True]).head(100))

Explicit baseline variables: ['debt_capacity_score_0_100', 'employment_strength_score_0_100', 'rd_intensity_score_0_100', 'human_capital_tertiary_score_0_100', 'energy_security_score_0_100', 'governance_capacity_score_0_100']
Review sample by shock:


,shock_id,countries
0,2007,42
1,2019,42


Baseline complete-case sample by shock:


,shock_id,countries
0,2007,25
1,2019,32


,Country,country_name,shock_id,analysis_year,Recovery,baseline_candidate_variables_total,baseline_candidate_variables_available,baseline_candidate_coverage_rate,baseline_complete_case,has_recovery,recommended_for_pre_poset_review,recommended_for_baseline_complete_case_poset
6,AUT,Austria,2007,2007,2.0000,6,6,1.0000,True,True,True,True
8,BEL,Belgium,2007,2007,1.0000,6,6,1.0000,True,True,True,True
23,CAN,Canada,2007,2007,1.0000,6,6,1.0000,True,True,True,True
36,CZE,Czechia,2007,2007,5.0000,6,6,1.0000,True,True,True,True
37,DEU,Germany,2007,2007,2.0000,6,6,1.0000,True,True,True,True
38,DNK,Denmark,2007,2007,7.0000,6,6,1.0000,True,True,True,True
42,ESP,Spain,2007,2007,8.0000,6,6,1.0000,True,True,True,True
43,EST,Estonia,2007,2007,8.0000,6,6,1.0000,True,True,True,True
44,FIN,Finland,2007,2007,8.0000,6,6,1.0000,True,True,True,True
45,FRA,France,2007,2007,2.0000,6,6,1.0000,True,True,True,True


In [12]:
# ------------------------------------------------------
# PRELIMINARY VARIABLE SET RECOMMENDATIONS
# ------------------------------------------------------
# These are recommendations for the next notebook, not final methodological decisions.

def existing_cols(cols):
    return [col for col in cols if col in aligned.columns]

baseline_variable_set = existing_cols(EXPLICIT_BASELINE_6_VARIABLES)

baseline_plus_gdp_stability_2007 = existing_cols(
    baseline_variable_set + ["gdp_growth_stability_pre_2007_score_0_100"]
)

baseline_plus_gdp_stability_2019 = existing_cols(
    baseline_variable_set + ["gdp_growth_stability_pre_2019_score_0_100"]
)

sensitivity_remove_governance = [
    col for col in baseline_variable_set
    if col != "governance_capacity_score_0_100"
]

sensitivity_replace_debt_with_productivity = existing_cols([
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
    "governance_capacity_score_0_100",
    "productivity_capacity_score_0_100",
])

sensitivity_coverage_safe_4 = existing_cols([
    "employment_strength_score_0_100",
    "energy_security_score_0_100",
    "governance_capacity_score_0_100",
    "gdp_growth_stability_pre_2007_score_0_100",
    "gdp_growth_stability_pre_2019_score_0_100",
])

candidate_variable_sets = pd.DataFrame([
    {
        "set_name": "baseline_6_variables",
        "intended_use": "main_candidate",
        "shock_specific": False,
        "variables": ",".join(baseline_variable_set),
        "variable_count": len(baseline_variable_set),
        "note": "Balanced structural set across fiscal, labour, innovation, human capital, energy, and governance capacity.",
    },
    {
        "set_name": "baseline_plus_gdp_stability_2007",
        "intended_use": "shock_2007_sensitivity",
        "shock_specific": True,
        "variables": ",".join(baseline_plus_gdp_stability_2007),
        "variable_count": len(baseline_plus_gdp_stability_2007),
        "note": "Adds pre-2007 GDP growth stability. Use only for 2007 shock rows.",
    },
    {
        "set_name": "baseline_plus_gdp_stability_2019",
        "intended_use": "shock_2019_sensitivity",
        "shock_specific": True,
        "variables": ",".join(baseline_plus_gdp_stability_2019),
        "variable_count": len(baseline_plus_gdp_stability_2019),
        "note": "Adds pre-2019 GDP growth stability. Use only for 2019 shock rows.",
    },
    {
        "set_name": "sensitivity_remove_governance",
        "intended_use": "sensitivity",
        "shock_specific": False,
        "variables": ",".join(sensitivity_remove_governance),
        "variable_count": len(sensitivity_remove_governance),
        "note": "Tests dependence on derived governance composite.",
    },
    {
        "set_name": "sensitivity_replace_debt_with_productivity",
        "intended_use": "sensitivity",
        "shock_specific": False,
        "variables": ",".join(sensitivity_replace_debt_with_productivity),
        "variable_count": len(sensitivity_replace_debt_with_productivity),
        "note": "Tests whether productive capacity changes ordering structure.",
    },
])

candidate_variable_sets.to_csv(
    PROCESSED_DIR / "candidate_variable_sets.csv",
    index=False,
)

candidate_variable_sets.to_csv(
    DIAGNOSTICS_DIR / "candidate_variable_sets.csv",
    index=False,
)

display(candidate_variable_sets)

,set_name,intended_use,shock_specific,variables,variable_count,note
0,baseline_6_variables,main_candidate,False,"debt_capacity_score_0_100,employment_strength_...",6,"Balanced structural set across fiscal, labour,..."
1,baseline_plus_gdp_stability_2007,shock_2007_sensitivity,True,"debt_capacity_score_0_100,employment_strength_...",7,Adds pre-2007 GDP growth stability. Use only f...
2,baseline_plus_gdp_stability_2019,shock_2019_sensitivity,True,"debt_capacity_score_0_100,employment_strength_...",7,Adds pre-2019 GDP growth stability. Use only f...
3,sensitivity_remove_governance,sensitivity,False,"debt_capacity_score_0_100,employment_strength_...",5,Tests dependence on derived governance composite.
4,sensitivity_replace_debt_with_productivity,sensitivity,False,"employment_strength_score_0_100,rd_intensity_s...",6,Tests whether productive capacity changes orde...


In [13]:
# ------------------------------------------------------
# CREATE ANALYSIS-READY SNAPSHOT FILES
# ------------------------------------------------------

# Full direction-aligned snapshot.
analysis_ready_full = aligned.copy()

analysis_ready_full.to_csv(
    PROCESSED_DIR / "pre_poset_analysis_ready_snapshot_full.csv",
    index=False,
)

# Baseline candidate snapshot.
baseline_output_cols = [
    "Country",
    "country_name",
    "shock_id",
    "analysis_year",
    "pre_shock_window",
    "Recovery",
] + baseline_variable_set

baseline_output_cols = [
    col for col in baseline_output_cols
    if col in aligned.columns
]

pre_poset_baseline_candidate_snapshot = aligned[baseline_output_cols].copy()

pre_poset_baseline_candidate_snapshot.to_csv(
    PROCESSED_DIR / "pre_poset_baseline_candidate_snapshot.csv",
    index=False,
)

# Recommended review sample only.
review_keys = recommended_analysis_sample_by_shock[["Country", "shock_id"]].copy()
review_keys["shock_id"] = review_keys["shock_id"].astype(str)

pre_poset_baseline_candidate_review_sample = pre_poset_baseline_candidate_snapshot.merge(
    review_keys,
    on=["Country", "shock_id"],
    how="inner",
)

pre_poset_baseline_candidate_review_sample.to_csv(
    PROCESSED_DIR / "pre_poset_baseline_candidate_review_sample.csv",
    index=False,
)

complete_case_keys = baseline_complete_case_sample_by_shock[["Country", "shock_id"]].copy()
complete_case_keys["shock_id"] = complete_case_keys["shock_id"].astype(str)

pre_poset_baseline_candidate_complete_case_sample = pre_poset_baseline_candidate_snapshot.merge(
    complete_case_keys,
    on=["Country", "shock_id"],
    how="inner",
)

pre_poset_baseline_candidate_complete_case_sample.to_csv(
    PROCESSED_DIR / "pre_poset_baseline_candidate_complete_case_sample.csv",
    index=False,
)

print("Full analysis-ready snapshot rows:", len(analysis_ready_full))
print("Baseline candidate snapshot rows:", len(pre_poset_baseline_candidate_snapshot))
print("Review sample rows:", len(pre_poset_baseline_candidate_review_sample))
print("Baseline complete-case sample rows:", len(pre_poset_baseline_candidate_complete_case_sample))

display(pre_poset_baseline_candidate_snapshot.head())
display(pre_poset_baseline_candidate_review_sample.head())
display(pre_poset_baseline_candidate_complete_case_sample.head())

Full analysis-ready snapshot rows: 300
Baseline candidate snapshot rows: 300
Review sample rows: 84
Baseline complete-case sample rows: 57


,Country,country_name,shock_id,analysis_year,pre_shock_window,Recovery,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,governance_capacity_score_0_100
0,AGO,Angola,2007,2007,2000-2007,NaN,NaN,NaN,NaN,NaN,75.6980,45.1550
1,ALB,Albania,2007,2007,2000-2007,NaN,NaN,NaN,NaN,NaN,14.2833,60.6684
2,ARE,United Arab Emirates,2007,2007,2000-2007,NaN,NaN,NaN,NaN,NaN,32.8476,46.6278
3,ARG,Argentina,2007,2007,2000-2007,NaN,NaN,NaN,6.7435,NaN,18.0225,33.0154
4,ARM,Armenia,2007,2007,2000-2007,NaN,NaN,NaN,NaN,NaN,12.7737,54.8547


,Country,country_name,shock_id,analysis_year,pre_shock_window,Recovery,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,governance_capacity_score_0_100
0,AUS,Australia,2007,2007,2000-2007,0.0000,85.7781,94.1485,NaN,65.0125,25.6804,99.6673
1,AUT,Austria,2007,2007,2000-2007,2.0000,45.5179,92.6786,54.7887,44.0754,12.9434,94.2334
2,BEL,Belgium,2007,2007,2000-2007,1.0000,26.4183,84.6671,40.5810,61.0357,11.5940,90.3204
3,BRA,Brazil,2007,2007,2000-2007,1.0000,NaN,82.7628,NaN,6.6077,16.7668,62.1695
4,CAN,Canada,2007,2007,2000-2007,1.0000,68.9667,88.6913,41.8854,100.0000,20.9717,88.1037


,Country,country_name,shock_id,analysis_year,pre_shock_window,Recovery,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,governance_capacity_score_0_100
0,AUT,Austria,2007,2007,2000-2007,2.0000,45.5179,92.6786,54.7887,44.0754,12.9434,94.2334
1,BEL,Belgium,2007,2007,2000-2007,1.0000,26.4183,84.6671,40.5810,61.0357,11.5940,90.3204
2,CAN,Canada,2007,2007,2000-2007,1.0000,68.9667,88.6913,41.8854,100.0000,20.9717,88.1037
3,CZE,Czechia,2007,2007,2000-2007,5.0000,79.4042,91.2674,27.1111,16.5599,15.8418,78.2822
4,DEU,Germany,2007,2007,2000-2007,2.0000,47.3662,81.0341,54.4377,42.1554,13.5621,82.0507


In [14]:
# ------------------------------------------------------
# RECOVERY ASSOCIATION DIAGNOSTICS
# ------------------------------------------------------
# This is validation-oriented diagnostics only.
# Recovery is not an ordering variable.

recovery_diag_rows = []

for shock_id, group in aligned.groupby("shock_id"):
    if "Recovery" not in group.columns:
        continue

    for _, meta in candidate_variable_inventory.iterrows():
        col = meta["candidate_variable"]

        if col not in group.columns:
            continue

        temp = group[[col, "Recovery"]].dropna().copy()

        if len(temp) >= 5:
            corr = temp[col].corr(temp["Recovery"])
        else:
            corr = np.nan

        recovery_diag_rows.append({
            "shock_id": shock_id,
            "candidate_variable": col,
            "concept": meta["concept"],
            "domain": meta["domain"],
            "countries_used": len(temp),
            "correlation_with_recovery": corr,
            "interpretation_note": "Diagnostic only; not causal evidence and not used for POSet ordering.",
        })

candidate_recovery_association_diagnostics = pd.DataFrame(recovery_diag_rows)

candidate_recovery_association_diagnostics.to_csv(
    DIAGNOSTICS_DIR / "candidate_recovery_association_diagnostics.csv",
    index=False,
)

display(candidate_recovery_association_diagnostics.sort_values(["shock_id", "correlation_with_recovery"]))

,shock_id,candidate_variable,concept,domain,countries_used,correlation_with_recovery,interpretation_note
4,2007,debt_capacity_score_0_100,Public debt capacity,fiscal_capacity,31,-0.3184,Diagnostic only; not causal evidence and not u...
6,2007,energy_security_score_0_100,Energy security,external_energy_resilience,43,-0.2196,Diagnostic only; not causal evidence and not u...
0,2007,rd_intensity_score_0_100,R&D intensity,innovation_capacity,38,-0.1416,Diagnostic only; not causal evidence and not u...
1,2007,human_capital_tertiary_score_0_100,Adult tertiary educational attainment,human_capital_capacity,37,-0.1168,Diagnostic only; not causal evidence and not u...
14,2007,productivity_stability_pre_2007_score_0_100,Productivity stability,productive_capacity,37,-0.1045,Diagnostic only; not causal evidence and not u...
12,2007,unemployment_stability_pre_2007_score_0_100,Unemployment stability,labour_market_capacity,42,-0.0943,Diagnostic only; not causal evidence and not u...
2,2007,productivity_capacity_score_0_100,GDP per hour worked,productive_capacity,37,0.0159,Diagnostic only; not causal evidence and not u...
7,2007,price_stability_level_score_0_100,Price stability level,macro_stability,44,0.0210,Diagnostic only; not causal evidence and not u...
8,2007,gdp_growth_stability_pre_2007_score_0_100,GDP growth stability,macro_stability,44,0.1131,Diagnostic only; not causal evidence and not u...
5,2007,employment_strength_score_0_100,Employment strength,labour_market_capacity,42,0.1328,Diagnostic only; not causal evidence and not u...


In [15]:
# ------------------------------------------------------
# OUTPUT DATA DICTIONARY AND METHODOLOGICAL NOTES
# ------------------------------------------------------

data_dictionary_rows = []

for _, row in candidate_variable_inventory.iterrows():
    data_dictionary_rows.append({
        "column": row["candidate_variable"],
        "raw_aligned_column": row["candidate_variable_raw"],
        "source_column": row["source_column"],
        "concept": row["concept"],
        "domain": row["domain"],
        "direction": "higher_is_better",
        "role": "candidate_poset_ordering_variable",
        "note": row["reason"],
    })

for col in validation_cols_present:
    data_dictionary_rows.append({
        "column": col,
        "raw_aligned_column": "",
        "source_column": col,
        "concept": "Observed recovery outcome",
        "domain": "validation_outcome",
        "direction": "lower_recovery_time_is_better_but_not_used_for_ordering",
        "role": "validation_only_not_ordering",
        "note": "Must not be used as a POSet ordering variable.",
    })

pre_poset_data_dictionary = pd.DataFrame(data_dictionary_rows)

pre_poset_data_dictionary.to_csv(
    PROCESSED_DIR / "pre_poset_data_dictionary.csv",
    index=False,
)

pre_poset_data_dictionary.to_csv(
    DIAGNOSTICS_DIR / "pre_poset_data_dictionary.csv",
    index=False,
)

methodological_notes = pd.DataFrame([
    {
        "topic": "Direction alignment",
        "note": "All candidate ordering variables are transformed so higher = better.",
    },
    {
        "topic": "Debt capacity",
        "note": "Public debt/GDP is lower-is-better, so debt capacity is defined as the negative of public debt/GDP and scaled within shock.",
    },
    {
        "topic": "Employment strength",
        "note": "Unemployment is lower-is-better, so employment strength is defined as the negative unemployment rate and scaled within shock.",
    },
    {
        "topic": "Energy security",
        "note": "Energy import dependency is lower-is-better. Negative raw dependency values for net exporters become high energy security after inversion.",
    },
    {
        "topic": "Governance",
        "note": "Governance capacity uses the project-derived WGI Mahalanobis composite, not the six WGI dimensions as separate main ordering variables.",
    },
    {
        "topic": "Human capital",
        "note": "Tertiary education is adult tertiary educational attainment age 25-64, representing accumulated human-capital stock.",
    },
    {
        "topic": "Recovery leakage",
        "note": "Recovery is kept only as a validation outcome and is excluded from candidate ordering variable inventory.",
    },
    {
        "topic": "Variable count",
        "note": "The baseline recommendation is around 6 variables. Avoid adding too many correlated variables before POSet analysis.",
    },
])

methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "pre_poset_methodological_notes.csv",
    index=False,
)

display(pre_poset_data_dictionary)
display(methodological_notes)

,column,raw_aligned_column,source_column,concept,domain,direction,role,note
0,rd_intensity_score_0_100,rd_intensity_aligned_raw,rd_gdp,R&D intensity,innovation_capacity,higher_is_better,candidate_poset_ordering_variable,R&D/GDP captures innovation capacity and is al...
1,human_capital_tertiary_score_0_100,human_capital_tertiary_aligned_raw,tertiary_education_25_64,Adult tertiary educational attainment,human_capital_capacity,higher_is_better,candidate_poset_ordering_variable,Adult tertiary attainment captures accumulated...
2,productivity_capacity_score_0_100,productivity_capacity_aligned_raw,productivity_gdp_per_hour,GDP per hour worked,productive_capacity,higher_is_better,candidate_poset_ordering_variable,"Productivity captures economic efficiency, but..."
3,governance_capacity_score_0_100,governance_capacity_aligned_raw,wgi_mahalanobis_ideal_score_full_wgi,Governance capacity,institutional_capacity,higher_is_better,candidate_poset_ordering_variable,Derived WGI governance composite summarizes in...
4,debt_capacity_score_0_100,debt_capacity_aligned_raw,public_debt_gdp_canonical,Public debt capacity,fiscal_capacity,higher_is_better,candidate_poset_ordering_variable,Lower public debt/GDP indicates greater fiscal...
5,employment_strength_score_0_100,employment_strength_aligned_raw,unemployment_rate,Employment strength,labour_market_capacity,higher_is_better,candidate_poset_ordering_variable,Lower unemployment indicates stronger labour-m...
6,energy_security_score_0_100,energy_security_aligned_raw,energy_import_dependency_raw,Energy security,external_energy_resilience,higher_is_better,candidate_poset_ordering_variable,Lower energy import dependency means higher en...
7,price_stability_level_score_0_100,price_stability_level_aligned_raw,inflation_cpi,Price stability level,macro_stability,higher_is_better,candidate_poset_ordering_variable,Inflation level is converted via negative abso...
8,gdp_growth_stability_pre_2007_score_0_100,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_negative_std_pre_2007,GDP growth stability,macro_stability,higher_is_better,candidate_poset_ordering_variable,Pre-shock GDP growth volatility converted to s...
9,gdp_growth_stability_pre_2019_score_0_100,gdp_growth_stability_pre_2019_aligned_raw,gdp_growth_stability_negative_std_pre_2019,GDP growth stability,macro_stability,higher_is_better,candidate_poset_ordering_variable,Pre-shock GDP growth volatility converted to s...


,topic,note
0,Direction alignment,All candidate ordering variables are transform...
1,Debt capacity,"Public debt/GDP is lower-is-better, so debt ca..."
2,Employment strength,"Unemployment is lower-is-better, so employment..."
3,Energy security,Energy import dependency is lower-is-better. N...
4,Governance,Governance capacity uses the project-derived W...
5,Human capital,Tertiary education is adult tertiary education...
6,Recovery leakage,Recovery is kept only as a validation outcome ...
7,Variable count,The baseline recommendation is around 6 variab...


In [16]:
# ------------------------------------------------------
# CREATE PRE-POSET EDA AUDIT WORKBOOK
# ------------------------------------------------------

PRE_POSET_AUDIT_XLSX = AUDIT_DIR / "06_Pre_POSet_EDA_Checks_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_snapshot_summary",
        "description": "Basic input snapshot summary.",
        "path": DIAGNOSTICS_DIR / "snapshot_basic_summary.csv",
    },
    {
        "sheet_name": "02_input_columns",
        "description": "Input snapshot columns and missingness.",
        "path": DIAGNOSTICS_DIR / "input_snapshot_columns.csv",
    },
    {
        "sheet_name": "03_direction_metadata",
        "description": "Direction alignment metadata.",
        "path": DIAGNOSTICS_DIR / "variable_direction_alignment_metadata.csv",
    },
    {
        "sheet_name": "04_direction_sanity",
        "description": "Direction alignment sanity checks.",
        "path": DIAGNOSTICS_DIR / "direction_alignment_sanity_checks.csv",
    },
    {
        "sheet_name": "05_candidate_inventory",
        "description": "Candidate ordering variable inventory.",
        "path": DIAGNOSTICS_DIR / "candidate_ordering_variable_inventory.csv",
    },
    {
        "sheet_name": "06_missingness",
        "description": "Candidate variable missingness by shock.",
        "path": DIAGNOSTICS_DIR / "candidate_variable_missingness_by_shock.csv",
    },
    {
        "sheet_name": "07_redundancy_pairs",
        "description": "Candidate variable redundancy pairs with absolute correlation >= 0.70.",
        "path": DIAGNOSTICS_DIR / "candidate_variable_redundancy_pairs.csv",
    },
    {
        "sheet_name": "08_scorecard",
        "description": "Candidate variable scorecard.",
        "path": DIAGNOSTICS_DIR / "candidate_variable_scorecard.csv",
    },
    {
        "sheet_name": "09_country_coverage",
        "description": "Country coverage scorecard.",
        "path": DIAGNOSTICS_DIR / "country_coverage_scorecard.csv",
    },
    {
        "sheet_name": "10_review_sample",
        "description": "Recommended analysis sample by shock for review.",
        "path": DIAGNOSTICS_DIR / "recommended_analysis_sample_by_shock.csv",
    },
    {
        "sheet_name": "11_complete_case",
        "description": "Baseline complete-case sample by shock.",
        "path": DIAGNOSTICS_DIR / "baseline_complete_case_sample_by_shock.csv",
    },
    {
        "sheet_name": "12_variable_sets",
        "description": "Candidate variable set recommendations.",
        "path": DIAGNOSTICS_DIR / "candidate_variable_sets.csv",
    },
    {
        "sheet_name": "13_recovery_diag",
        "description": "Diagnostic association with recovery outcomes.",
        "path": DIAGNOSTICS_DIR / "candidate_recovery_association_diagnostics.csv",
    },
    {
        "sheet_name": "14_dictionary",
        "description": "Pre-POSet data dictionary.",
        "path": DIAGNOSTICS_DIR / "pre_poset_data_dictionary.csv",
    },
    {
        "sheet_name": "15_method_notes",
        "description": "Methodological notes.",
        "path": DIAGNOSTICS_DIR / "pre_poset_methodological_notes.csv",
    },
]

used_sheets = set()

with pd.ExcelWriter(PRE_POSET_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "06 Pre-POSet EDA Checks Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for direction alignment, redundancy checks, and candidate variable selection diagnostics.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Pre-POSet EDA audit workbook created:")
print(PRE_POSET_AUDIT_XLSX.resolve())

Pre-POSet EDA audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\06_Pre_POSet_EDA_Checks_Audit.xlsx


In [17]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("06 PRE-POSET EDA CHECKS COMPLETE")
print("=" * 80)

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nAudit workbook:")
print(PRE_POSET_AUDIT_XLSX.resolve())

print("\nMain processed outputs:")
main_outputs = [
    "pre_poset_structural_snapshot_direction_aligned.csv",
    "candidate_ordering_variable_inventory.csv",
    "candidate_variable_scorecard.csv",
    "country_coverage_scorecard.csv",
    "recommended_analysis_sample_by_shock.csv",
    "candidate_variable_sets.csv",
    "pre_poset_analysis_ready_snapshot_full.csv",
    "pre_poset_baseline_candidate_snapshot.csv",
    "pre_poset_baseline_candidate_review_sample.csv",
    "pre_poset_baseline_candidate_complete_case_sample.csv",
    "baseline_complete_case_sample_by_shock.csv",
    "pre_poset_data_dictionary.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nRecommended baseline variable set:")
display(candidate_variable_sets[candidate_variable_sets["set_name"] == "baseline_6_variables"])

print("\nCandidate variable scorecard:")
display(candidate_variable_scorecard)

print("\nImportant notes:")
print("1. Candidate variables are direction-aligned so higher = better.")
print("2. Recovery remains validation-only and is not included in ordering variable inventory.")
print("3. Correlation/redundancy checks were created but final variable decisions happen before POSet execution.")
print("4. The baseline candidate set targets around 6 variables.")
print("5. The next notebook should run profile-level POSet analysis using selected variable sets.")

print("\nNext notebook:")
print("07_Profile_POSet_Main.ipynb")

06 PRE-POSET EDA CHECKS COMPLETE

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\06_Pre_POSet_EDA_Checks

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\06_Pre_POSet_EDA_Checks_Audit.xlsx

Main processed outputs:
- FOUND: pre_poset_structural_snapshot_direction_aligned.csv
- FOUND: candidate_ordering_variable_inventory.csv
- FOUND: candidate_variable_scorecard.csv
- FOUND: country_coverage_scorecard.csv
- FOUND: recommended_analysis_sample_by_shock.csv
- FOUND: candidate_variable_sets.csv
- FOUND: pre_poset_analysis_ready_snapshot_full.csv
- FOUND: pre_poset_baseline_candidate_snapshot.csv
- FOUND: pre_poset_baseline_candidate_review_s

,set_name,intended_use,shock_specific,variables,variable_count,note
0,baseline_6_variables,main_candidate,False,"debt_capacity_score_0_100,employment_strength_...",6,"Balanced structural set across fiscal, labour,..."



Candidate variable scorecard:


,candidate_variable,candidate_variable_raw,source_column,concept,domain,priority,direction,reason,recommended_use,min_coverage_rate_broad_master,mean_coverage_rate_broad_master,min_countries_non_missing_broad_master,min_analysis_coverage_rate,mean_analysis_coverage_rate,min_analysis_countries_non_missing,shocks_available,max_abs_correlation,max_redundancy_class,pre_poset_recommendation,warning
0,energy_security_score_0_100,energy_security_aligned_raw,energy_import_dependency_raw,Energy security,external_energy_resilience,baseline_candidate,higher_is_better,Lower energy import dependency means higher en...,main_or_baseline,0.8896,0.8900,130,0.9767,0.9770,42,2,0.3162,lower_redundancy_risk,recommended_baseline_pool,
1,human_capital_tertiary_score_0_100,human_capital_tertiary_aligned_raw,tertiary_education_25_64,Adult tertiary educational attainment,human_capital_capacity,baseline_candidate,higher_is_better,Adult tertiary attainment captures accumulated...,main_or_baseline,0.2534,0.2728,37,0.8409,0.8972,37,2,0.6869,lower_redundancy_risk,recommended_baseline_pool,
2,rd_intensity_score_0_100,rd_intensity_aligned_raw,rd_gdp,R&D intensity,innovation_capacity,baseline_candidate,higher_is_better,R&D/GDP captures innovation capacity and is al...,main_or_baseline,0.2922,0.2968,44,0.8636,0.8737,38,2,0.6869,lower_redundancy_risk,recommended_baseline_pool,
3,governance_capacity_score_0_100,governance_capacity_aligned_raw,wgi_mahalanobis_ideal_score_full_wgi,Governance capacity,institutional_capacity,baseline_candidate,higher_is_better,Derived WGI governance composite summarizes in...,main_or_baseline,0.9863,0.9932,144,1.0000,1.0000,43,2,0.5497,lower_redundancy_risk,recommended_baseline_pool,
4,employment_strength_score_0_100,employment_strength_aligned_raw,unemployment_rate,Employment strength,labour_market_capacity,baseline_candidate,higher_is_better,Lower unemployment indicates stronger labour-m...,main_or_baseline,0.3356,0.3366,49,0.9545,0.9656,42,2,0.4342,lower_redundancy_risk,recommended_baseline_pool,
5,gdp_growth_stability_pre_2007_score_0_100,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_negative_std_pre_2007,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline,0.3014,0.3014,44,1.0000,1.0000,44,1,0.6874,lower_redundancy_risk,recommended_baseline_pool,
6,gdp_growth_stability_pre_2019_score_0_100,gdp_growth_stability_pre_2019_aligned_raw,gdp_growth_stability_negative_std_pre_2019,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline,0.2857,0.2857,44,1.0000,1.0000,43,1,0.7370,moderate_redundancy_risk,recommended_baseline_pool,
7,unemployment_stability_pre_2007_score_0_100,unemployment_stability_pre_2007_aligned_raw,unemployment_stability_negative_std_pre_2007,Unemployment stability,labour_market_capacity,review_candidate,higher_is_better,Pre-shock unemployment volatility converted to...,review_or_sensitivity,0.3288,0.3288,48,0.9545,0.9545,42,1,0.4128,lower_redundancy_risk,review_candidate,
8,unemployment_stability_pre_2019_score_0_100,unemployment_stability_pre_2019_aligned_raw,unemployment_stability_negative_std_pre_2019,Unemployment stability,labour_market_capacity,review_candidate,higher_is_better,Pre-shock unemployment volatility converted to...,review_or_sensitivity,0.3312,0.3312,51,0.9535,0.9535,41,1,0.6411,lower_redundancy_risk,review_candidate,
9,inflation_stability_pre_2007_score_0_100,inflation_stability_pre_2007_aligned_raw,inflation_stability_negative_std_pre_2007,Inflation stability,macro_stability,review_candidate,higher_is_better,Pre-shock inflation volatility converted to st...,review_or_sensitivity,0.3219,0.3219,47,1.0000,1.0000,44,1,0.6874,lower_redundancy_risk,review_candidate,



Important notes:
1. Candidate variables are direction-aligned so higher = better.
2. Recovery remains validation-only and is not included in ordering variable inventory.
3. Correlation/redundancy checks were created but final variable decisions happen before POSet execution.
4. The baseline candidate set targets around 6 variables.
5. The next notebook should run profile-level POSet analysis using selected variable sets.

Next notebook:
07_Profile_POSet_Main.ipynb
